# **2. Кластеризация данных (Clustering Analysis): Сегментация пациентов и поиск скрытых паттернов**

* __Цель кластеризации:__ Произвести объективное разбиение совокупности пациентов на однородные клинические группы (кластеры) на основе многомерных медицинских показателей, используя методы машинного обучения без учителя (Unsupervised Learning).
* __Задачи кластеризации:__ Выявить скрытую топологию медицинских данных без использования заранее заданных меток (диагнозов). Сравнить эффективность различных парадигм кластеризации (центроидные, плотностные, иерархические алгоритмы). Математически обосновать оптимальное количество кластеров и сформировать клинические профили групп риска для последующего предиктивного анализа.
* __Алгоритм использования:__
  1. __Подготовка признакового пространства:__ Извлечение матрицы признаков из главных компонент (PCA) с сокрытием целевой переменной для обеспечения строгой "слепой" сегментации.
  2. __Кластеризация K-Means (Центроидный подход):__ Определение оптимального числа кластеров с помощью метода локтя (Elbow Method) и коэффициента силуэта (Silhouette Score), обучение модели и базовая бинарная сегментация.
  3. __Кластеризация DBSCAN (Плотностной подход):__ Расчет матрицы k-расстояний для определения радиуса окрестности (`eps`) и фильтрации плотных ядер кластеров от переходных (шумовых) состояний.
  4. __Иерархическая кластеризация (Агломеративный подход):__ Построение матрицы связей (метод Уорда) и дендрограммы для визуального определения уровня отсечения, выявление промежуточной "серой зоны".
  5. __Сравнительная оценка и интерпретация:__ Сопоставление алгоритмов, формирование итоговых сегментов (низкий, умеренный и высокий риск) и сохранение размеченного датасета для статистических тестов.

In [ ]:
# --- ИМПОРТ БИБЛИОТЕК И ПЕРВИЧНАЯ НАСТРОЙКА ---

# Сторонние библиотеки
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# Локальные модули
from scripts.logger import setup_logger
from scripts.utils import save_dataset

# Инициализация логгера
logger = setup_logger("clustering")

# Установка параметра для воспроизводимости результатов
RANDOM_STATE = 42

# Настройка визуализации
sns.set_theme(
    style="whitegrid",
    palette="muted",
    rc={"figure.figsize": (10, 6), "axes.titlesize": 14},
)

In [ ]:
# --- ЗАГРУЗКА ДАННЫХ ---

PROCESSED_DIR = Path("../data/processed")
INTERIM_DIR = Path("../data/interim")

data_pca = pd.read_csv(PROCESSED_DIR / "heart_disease_uci_pca.csv")
data_original = pd.read_csv(INTERIM_DIR / "heart_disease_uci_cleaned.csv")

## **2.1. Подготовка данных к кластеризации**

* __Цель:__ Сформировать корректную, очищенную матрицу признаков для объективного обучения алгоритмов кластеризации "без учителя".
* __Задачи:__ Обеспечить «слепую» сегментацию данных путем исключения целевой переменной для предотвращения утечки информации при поиске скрытых паттернов. Провести верификацию целостности итоговой матрицы на предмет отсутствия пропусков после этапа предобработки. Выполнить контрольную оценку описательной статистики главных компонент (PCA) для подтверждения качества масштабирования данных.
* __Алгоритм использования:__
    1. __Сепарация данных:__ Отделение признакового пространства ($X$) от целевой переменной (`num`) и системных идентификаторов (`id`).
    2. __Валидация матрицы:__ Проверка на отсутствие значений `NaN` в сформированной матрице признаков.
    3. __Статистический аудит:__ Проверка центрирования данных (среднее значение должно стремиться к нулю) с помощью метода `describe()`.

In [ ]:
# --- 2.1.1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---


def analyze_cluster_profiles(data, labels, cluster_name="Cluster"):
    """
    Функция для анализа профилей кластеров, включая количественное распределение и средние значения медицинских показателей.
    Параметры:
    - data: DataFrame с исходными данными.
    - labels: Массив меток кластеров для каждого пациента.
    - cluster_name: Название столбца для кластеров в результирующем DataFrame.
    Возвращает:
    - size_df: DataFrame с количеством пациентов и долей для каждого кластера.
    - cluster_means: DataFrame со средними значениями медицинских показателей для каждого кластера.
    """
    # Создание копии оригинальных данных
    df_profile = data.copy()
    df_profile[cluster_name] = labels

    # Расчет размеров кластеров
    cluster_sizes = df_profile[cluster_name].value_counts().sort_index()
    cluster_percentages = (cluster_sizes / len(df_profile) * 100).round(1)

    size_df = (
        pd.DataFrame(
            {"Количество пациентов": cluster_sizes, "Доля (%)": cluster_percentages}
        )
        .reset_index()
        .rename(columns={"index": "Кластер"})
    )

    display(Markdown(f"* **Количественное распределение ({cluster_name}):**"))
    display(size_df)

    # Исключение неинформативных столбцов перед расчетом средних значений признаков
    cols_to_drop = ["id", "cluster"]
    existing_drops = [col for col in cols_to_drop if col in df_profile.columns]

    # Расчет средних значений признаков для каждого кластера
    cluster_means = (
        df_profile.drop(columns=existing_drops)
        .groupby(cluster_name)
        .mean(numeric_only=True)
        .round(2)
    )

    display(
        Markdown(f"* **Средние значения медицинских показателей ({cluster_name}):**")
    )
    display(cluster_means.T)

    return size_df, cluster_means

In [ ]:
# --- 2.1.2. ПОДГОТОВКА МАТРИЦЫ ПРИЗНАКОВ ---

display(Markdown("#### **Формирование матрицы X**"))

# Отделение признаков (X) от целевой переменной / меток (y)
X_cluster = data_pca.drop(columns=["num"]).copy()
y_true = data_pca["num"].copy()

# Проверка отсутствия пропусков и корректности данных
missing_values = X_cluster.isna().sum().sum()

display(
    Markdown(
        f"* Матрица признаков `X` успешно сформирована.\n"
        f"* Размерность матрицы признаков `X`: **{X_cluster.shape[0]} строк и {X_cluster.shape[1]} главных компонент (PC)**.\n"
        f"* Пропущенных значений в данных PCA: **{missing_values}**."
    )
)

# Проверка распределения (PCA-компоненты должны иметь среднее ~0)
display(Markdown("#### **Описательная статистика признаков (X_cluster)**"))
display(X_cluster.describe().round(4))

## **2.2. Кластеризация методом K-Means**

### **2.2.1. Определение оптимального числа кластеров**

* __Цель:__ Определить математически обоснованное и практически интерпретируемое количество кластеров (K) для обучения модели K-Means.
* __Задачи:__ Рассчитать внутрикластерную дисперсию для различных значений K. Вычислить коэффициент силуэта для оценки плотности и обособленности групп. Визуализировать метрики и выбрать K на основе математического консенсуса.
* __Алгоритм использования:__
  1. __Метод локтя (Elbow Method):__ Оценивает сумму квадратов расстояний от точек до центроидов (инерцию). Оптимальным считается значение K, при котором на графике образуется характерный изгиб ("локоть"), а скорость уменьшения ошибки резко падает.
  2. __Коэффициент силуэта (Silhouette Score):__ Измеряет плотность кластеров и расстояние между ними (от -1 до 1). Чем ближе к 1, тем лучше сгруппированы объекты. Оптимальным считается K, при котором коэффициент достигает глобального максимума.

In [ ]:
# --- 2.2.1.1. РАСЧЕТ МЕТРИК ---

# Списки для сохранения метрик
inertia_values = []
silhouette_scores = []

# Диапазон значений K
K_range = range(1, 11)
K_range_sil = range(2, 11)

display(
    Markdown("#### *Вычисление метрик (Инерция и Силуэт) для различных значений K...*")
)

for k in K_range:
    # Инициализация и обучение модели
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(X_cluster)

    # Сохранение инерции (Метод локтя)
    inertia_values.append(kmeans.inertia_)

    # Расчет коэффициента силуэта (K > 1)
    if k > 1:
        silhouette_avg = silhouette_score(X_cluster, kmeans.labels_)
        silhouette_scores.append(silhouette_avg)

# --- 2.2.1.2. ВИЗУАЛИЗАЦИЯ ---

plt.figure(figsize=(18, 6))

# График 1: Метод локтя (Elbow Method)
plt.subplot(1, 2, 1)
sns.lineplot(
    x=list(K_range),
    y=inertia_values,
    marker="o",
    color="teal",
    linewidth=2,
    markersize=8,
)
plt.title("Метод локтя (Elbow Method)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Инерция (Inertia)", fontsize=12)
plt.xticks(list(K_range))
plt.grid(True, linestyle="--", alpha=0.6)

# График 2: Коэффициент силуэта (Silhouette Score)
plt.subplot(1, 2, 2)
sns.lineplot(
    x=list(K_range_sil),
    y=silhouette_scores,
    marker="s",
    color="coral",
    linewidth=2,
    markersize=8,
)
plt.title("Коэффициент силуэта (Silhouette Score)", fontsize=14, pad=10)
plt.xlabel("Количество кластеров (K)", fontsize=12)
plt.ylabel("Средний коэффициент силуэта", fontsize=12)
plt.xticks(list(K_range_sil))
plt.grid(True, linestyle="--", alpha=0.6)

# Автоматическое нахождение и выделение максимума на графике силуэта
max_sil_index = np.argmax(silhouette_scores)
optimal_k_sil = K_range_sil[max_sil_index]
max_sil_value = silhouette_scores[max_sil_index]

plt.plot(optimal_k_sil, max_sil_value, marker="*", markersize=18, color="crimson")
plt.annotate(
    f" Максимум\n (K={optimal_k_sil})",
    xy=(optimal_k_sil, max_sil_value),
    xytext=(optimal_k_sil + 0.3, max_sil_value - 0.01),
    fontsize=12,
    color="darkred",
    weight="bold",
)

plt.tight_layout()
plt.show()

#### **Вывод по определению оптимального числа кластеров**

* **Анализ инерции (Метод локтя):** На левом графике наиболее выраженный излом ("локоть") наблюдается в диапазоне $K=2$ и $K=3$. Начиная с $K=3$, скорость снижения внутрикластерной дисперсии существенно падает, что говорит о нецелесообразности дальнейшего дробления данных.
* **Анализ плотности (Коэффициент силуэта):** Правый график демонстрирует однозначный глобальный максимум при $K=2$ (коэффициент силуэта $\approx 0.177$). Это математически подтверждает, что разбиение на две группы обеспечивает наилучшую сепарацию объектов в многомерном пространстве.
* **Итоговое обоснование:** Основываясь на консенсусе обеих метрик, для обучения алгоритма K-Means зафиксировано значение **$K=2$**. Данное математическое решение имеет сильный медицинский смысл в контексте данного набора данных: алгоритм обучения без учителя самостоятельно обнаружил скрытое бинарное разделение пациентов на две ключевые когорты — условную "группу нормы" и "группу риска".

### **2.2.2. Применение алгоритма K-Means**

* __Цель:__ Выполнить сегментацию пациентов на основе выбранного оптимального значения $K$.
* __Задачи:__ Инициализировать и обучить центроидную модель K-Means, используя математически обоснованные параметры и настройки воспроизводимости. Сгенерировать вектор принадлежности к кластерам для каждого объекта выборки. Обеспечить подготовку многомерных данных для пространственной визуализации в сниженном признаковом пространстве.
* __Алгоритм использования:__
    1. __Фиксация гиперпараметров:__ Установка оптимального количества кластеров (например, `OPTIMAL_K = 2`) и параметра `random_state`.
    2. __Обучение модели:__ Применение метода `fit()` к подготовленной матрице главных компонент.
    3. __Маркировка выборки:__ Извлечение меток кластеров через атрибут `labels_` и их интеграция в рабочий датасет.

In [ ]:
# --- 2.2.2.1. ОБУЧЕНИЕ МОДЕЛИ ---

display(Markdown("#### **Обучение модели K-Means**"))

OPTIMAL_K = optimal_k_sil

# Инициализация и обучение модели
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
kmeans_final.fit(X_cluster)

# Получение меток кластеров
cluster_labels = kmeans_final.labels_

# display(Markdown(f"* Модель успешно обучена. Получены метки для **{len(cluster_labels)}** объектов."))

display(
    Markdown(
        f"* Модель успешно обучена на **{OPTIMAL_K}** кластерах.\n"
        f"* Получены метки для **{len(cluster_labels)}** объектов."
    )
)

### **2.2.3. Визуализация и анализ**

* __Цель:__ Наглядно оценить качество разделения и сформировать детальные профили полученных групп пациентов.
* __Задачи:__ Проанализировать пространственную обособленность сегментов в 2D-проекции PCA с графическим выделением центроидов. Выполнить количественную оценку структуры полученных групп путем расчета объемов и процентных долей. Провести сопоставление средних клинических показателей в разрезе кластеров на базе исходных (не трансформированных) данных.
* __Алгоритм использования:__
    1. __Многомерная отрисовка:__ Построение `scatterplot` в осях PC1 и PC2 с цветовой кодировкой кластеров и наложением координат центроидов.
    2. __Количественный анализ:__ Расчет размеров сегментов с помощью функции `value_counts()`.
    3. __Профилирование:__ Группировка данных (`groupby`) по меткам кластеров для вычисления векторов средних значений клинических признаков.

In [ ]:
# --- 2.2.3.1. ВИЗУАЛИЗАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Визуализация кластеров K-Means в 2D-пространстве**"))

# Создание копии X_cluster для графика и добавление меток
plot_df = X_cluster.copy()
plot_df["Cluster"] = cluster_labels.astype(str)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="Set1",
    data=plot_df,
    alpha=0.7,
    s=60,
    edgecolor="k",
)

# Добавление центроидов на график
centroids_pca = kmeans_final.cluster_centers_
plt.scatter(
    centroids_pca[:, 0],
    centroids_pca[:, 1],
    marker="X",
    s=100,
    c="black",
    label="Центроиды",
    edgecolor="white",
    linewidth=2,
)

plt.title("2D-проекция кластеров K-Means (PC1 vs PC2)", fontsize=14)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (K-Means)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.2.3.2. ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Анализ профилей сегментов (K-Means)**"))

size_df_kmeans, means_kmeans = analyze_cluster_profiles(
    data=data_original, labels=cluster_labels, cluster_name="Cluster_KMeans"
)

#### **Выводы и описание сегментов (K-Means)**

__1. Визуализация (Разделимость кластеров):__
* На 2D-проекции отчетливо видно, что алгоритм K-Means успешно разделил пациентов на два крупных облака.
* Основная дисперсия и разделение происходят вдоль оси первой главной компоненты (PC1). 
* Несмотря на естественное для медицинских данных пересечение (overlap) в пограничной зоне, центроиды кластеров (черные кресты) находятся на значительном расстоянии друг от друга, что подтверждает математически качественную группировку.

__2. Интерпретация (Медицинские профили пациентов):__
Анализ средних значений исходных признаков позволяет составить четкие клинические портреты полученных групп:

* **Кластер 0 (Группа высокого риска / Пациенты с патологиями):**
  * **Размер:** 516 пациентов (56.2%).
  * **Профиль:** Пациенты старшей возрастной группы (средний возраст ~59 лет).
  * **Симптоматика:** Наблюдается повышенное артериальное давление в покое (`trestbps` = 134.5). Максимальная частота сердечных сокращений при нагрузке заметно снижена (`thalch` = 126). 
  * **Критические маркеры:** Более половины пациентов (54%) испытывают стенокардию, вызванную физической нагрузкой (`exang`), а показатель депрессии сегмента ST (`oldpeak`) критически завышен (1.22).
  * **Валидация (Диагноз):** Фактический средний показатель тяжести болезни (`num` = 1.45). *Хотя этот признак был скрыт от алгоритма*, K-Means абсолютно верно собрал в данный кластер людей с подтвержденными сердечно-сосудистыми заболеваниями.

* **Кластер 1 (Группа низкого риска / Условно здоровые):**
  * **Размер:** 402 пациента (43.8%).
  * **Профиль:** Более молодые пациенты (средний возраст ~47 лет).
  * **Симптоматика:** Давление находится в пределах нормы (`trestbps` = 125.7). Сердечно-сосудистая система отлично справляется с нагрузками: средний максимальный пульс высокий (`thalch` = 153).
  * **Критические маркеры:** Лишь 14% пациентов отмечают стенокардию при нагрузке, показатель `oldpeak` минимален (0.25).
  * **Валидация (Диагноз):** Показатель `num` (0.41) подтверждает, что модель "вслепую" выделила группу преимущественно здоровых пациентов.

## **2.3. Кластеризация методом DBSCAN**

### **2.3.1. Подбор параметров**

* __Цель:__ Определить оптимальные гиперпараметры `eps` и `min_samples` для алгоритма пространственной кластеризации DBSCAN с учетом специфичной плотности медицинских данных.
* __Задачи:__ Вычислить параметр `min_samples` на основе размерности пространства. Построить график k-расстояний для оценки плотности точек. Выбрать оптимальный радиус окрестности `eps` для надежной изоляции плотных ядер пациентов.
* __Алгоритм использования:__
  1. **Определение `min_samples`:** Классически принимается как удвоенное количество признаков (размерность матрицы PCA). Минимальное количество точек для образования ядра кластера.
  2. **Определение `eps` (Метод k-расстояний):** Расчет расстояния до k-го соседа для каждой точки с последующей сортировкой. Точка изгиба на графике указывает на оптимальное максимальное расстояние между двумя точками-соседями.

In [ ]:
# --- 2.3.1.1. ОПРЕДЕЛЕНИЕ MIN_SAMPLES ---

display(Markdown("#### **Определение параметра eps**"))

# Рекомендация: n_features * 2
min_samples = X_cluster.shape[1] * 2
display(
    Markdown(
        f"* Выбрано значение `min_samples` (удвоенная размерность PCA): **{min_samples}**"
    )
)

# --- 2.3.1.2. МЕТОД K-РАССТОЯНИЙ ---

# Обучение модели NearestNeighbors для поиска расстояния до k-го соседа ((k = min_samples))
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_cluster)

# Получение матрицы расстояний
distances, indices = neighbors_fit.kneighbors(X_cluster)

# Сортировка расстояния по убыванию
dist_k = np.sort(distances[:, min_samples - 1])[::-1]

In [ ]:
# --- 2.3.1.3. ПОСТРОЕНИЕ ГРАФИКА K-РАССТОЯНИЙ ДЛЯ ПОДБОРА EPS ---

# Визуализация графика k-расстояний
display(Markdown("#### **График k-расстояний (k-distance graph)**"))

plt.figure(figsize=(12, 6))
plt.plot(dist_k, color="teal", linewidth=2.5)
plt.title(f"График k-расстояний (k={min_samples}) для определения eps", fontsize=14)
plt.xlabel("Точки данных, отсортированные по убыванию расстояния", fontsize=12)
plt.ylabel(f"Расстояние до {min_samples}-го соседа (eps)", fontsize=12)

plt.grid(True, which="both", linestyle="--", alpha=0.7)
plt.minorticks_on()
plt.grid(True, which="minor", linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()

#### **Анализ графика и обоснование выбора параметров**

* **Математический подход:** Визуально изгиб ("локоть") на графике k-расстояний начинается в диапазоне от **1.8 до 2.5**. По классической теории именно здесь следует искать оптимальный `eps`.
* **Специфика данных:** Однако, медицинские данные представляют собой непрерывный спектр с высокой плотностью в центре (переходная зона между здоровьем и патологией). Если выбрать классическое высокое значение (например, `eps = 2.3`), алгоритм DBSCAN сольет почти всех пациентов в один гигантский кластер из-за так называемого "цепного эффекта".
* **Решение:** Чтобы разорвать эти связи и заставить алгоритм выделить именно самые плотные ядра (явно выраженных больных и явно здоровых пациентов), параметр `eps` был эмпирически занижен. 
* **Итог:** Для обучения модели фиксируются параметры **`eps = 1.4`** и **`min_samples = 16`**. Это неизбежно увеличит количество "шумовых" точек (выбросов), но позволит получить логически интерпретируемые сегменты.

### **2.3.2. Применение алгоритма DBSCAN**

* __Цель:__ Реализовать плотностную кластеризацию и выделить наиболее устойчивые ядра групп пациентов, изолировав аномалии и «пограничные» случаи.
* __Задачи:__ Обучить модель DBSCAN с использованием параметров плотности, выявленных в ходе предварительного анализа. Провести аудит структуры данных, определив количество найденных ядерных групп и объем выявленных шумовых выбросов. Визуализировать топологию плотности, дифференцируя структурные кластеры и точки, не вошедшие в устойчивые объединения.
* __Алгоритм использования:__
    1. __Параметризация:__ Инициализация алгоритма с заданными константами `eps` и `min_samples`.
    2. __Исполнение кластеризации:__ Запуск метода `fit_predict()` для одновременного обучения и разметки данных.
    3. __Анализ структуры:__ Идентификация шума (метки `-1`) и расчет доли «выбросов» от общего объема выборки.

In [ ]:
# --- 2.3.2.1. ОБУЧЕНИЕ МОДЕЛИ DBSCAN ---

display(Markdown("#### **Обучение модели DBSCAN**"))

# Ручной выбор параметров на основе графика k-расстояний
EPS_VALUE = 1.4
MIN_SAMPLES_VALUE = 16

# Инициализация и обучение модели
dbscan = DBSCAN(eps=EPS_VALUE, min_samples=MIN_SAMPLES_VALUE)
dbscan_labels = dbscan.fit_predict(X_cluster)

# Подсчет количества кластеров и шумовых точек
n_clusters_ = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_ = list(dbscan_labels).count(-1)
noise_percent = (n_noise_ / len(X_cluster)) * 100

# --- 2.3.2.2. АНАЛИЗ РЕЗУЛЬТАТОВ ---

display(
    Markdown(
        f"* Заданные параметры: `eps` = **{EPS_VALUE}**, `min_samples` = **{MIN_SAMPLES_VALUE}**\n"
        f"* Количество найденных кластеров: **{n_clusters_}**\n"
        f"* Количество шумовых точек (выбросов): **{n_noise_}** ({noise_percent:.1f}% от всех данных)"
    )
)

In [ ]:
# --- 2.3.2.3. ВИЗУАЛИЗАЦИЯ ---

display(Markdown("#### **Визуализация кластеров DBSCAN в 2D-пространстве**"))

# Создание копии X_cluster для графика и добавление меток DBSCAN
plot_df_dbscan = X_cluster.copy()
plot_df_dbscan["Cluster_DBSCAN"] = dbscan_labels

plt.figure(figsize=(12, 6))

# Разделение данных для дифференцированной отрисовки
noise_data = plot_df_dbscan[plot_df_dbscan["Cluster_DBSCAN"] == -1]
core_data = plot_df_dbscan[plot_df_dbscan["Cluster_DBSCAN"] != -1]

# Отрисовка шумовых точек (серые крестики)
if not noise_data.empty:
    sns.scatterplot(
        x="PC1",
        y="PC2",
        data=noise_data,
        color="dimgrey",
        marker="x",
        s=30,
        label="Шум (-1)",
        alpha=0.7,
    )

# Отрисовка кластеров (яркие точки)
if not core_data.empty:
    core_data = core_data.copy()
    core_data["Cluster_DBSCAN"] = core_data["Cluster_DBSCAN"].astype(str)

    sns.scatterplot(
        x="PC1",
        y="PC2",
        hue="Cluster_DBSCAN",
        palette="tab10",
        data=core_data,
        s=60,
        alpha=0.7,
        edgecolor="k",
    )

plt.title(
    f"2D-проекция кластеров DBSCAN (eps={EPS_VALUE}, min_samples={MIN_SAMPLES_VALUE})",
    fontsize=14,
)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (DBSCAN)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### **Выводы и интерпретация результатов (DBSCAN)**

**1. Особенности плотностного разделения:**
* В отличие от K-Means, который принудительно разделил все точки жесткой границей, DBSCAN выделил только **самые плотные ядра** концентрации пациентов.
* Значительная часть данных (~40%) была отнесена к кластеру `-1` (шум). На визуализации отчетливо видно, что эти точки (серые маркеры) располагаются преимущественно в центре пространства, образуя "мост" между двумя основными цветными облаками.

**2. Медицинский смысл "шума" (Переходная зона):**
* В контексте кардиологических данных этот "шум" — это не системная ошибка или бракованные записи. Это пациенты, находящиеся в **переходной зоне (группа умеренного риска)**. 
* Их показатели уже начали отклоняться от идеальной нормы, но еще не достигли критических значений явной патологии. DBSCAN математически подтвердил, что переход от здоровья к болезни в наших данных — это непрерывный спектр, а не два изолированных состояния.

**3. Сравнительный анализ:**
* Если **K-Means** оптимален для агрессивной бинарной сегментации (охват 100% базы), то **DBSCAN** блестяще справляется с задачей выявления "идеальных" представителей каждой группы. 
* Алгоритм надежно изолировал явно здоровых пациентов и пациентов с тяжелой симптоматикой (ядра), оставив пограничные и сомнительные случаи (шум) для индивидуального или дополнительного врачебного изучения.

## **2.4. Иерархическая кластеризация**

### **2.4.1. Построение дендрограммы**

* __Цель:__ Исследовать внутреннюю иерархическую структуру медицинских показателей пациентов и визуально определить оптимальное количество кластеров.
* __Задачи:__ Сформировать матрицу связей методом Уорда для минимизации суммарного увеличения внутрикластерной дисперсии при объединении групп. Построить усеченную древовидную структуру (дендрограмму) для повышения читаемости иерархических связей. Определить математически обоснованный уровень отсечения (cut-off) дерева на основе анализа самых длинных вертикальных ветвей дерева.
* __Алгоритм использования:__
    1. __Матричный расчет:__ Вычисление связей через функцию `linkage` с метрикой `method="ward"`.
    2. __Дендрографический анализ:__ Отрисовка дерева с параметром `truncate_mode="level"` для визуализации верхних уровней иерархии.
    3. __Фиксация порога:__ Установка линии отсечения (`axhline`) на выбранной дистанции (например, `y=30`) для определения итогового числа кластеров.

In [ ]:
# --- 2.4.1.1. ВИЗУАЛИЗАЦИЯ ИЕРАРХИИ ---

display(Markdown("#### **Визуализация иерархии: Дендрограмма (Метод Варда)**"))

# 1. Расчет матрицы связей (Linkage Matrix)
# Метод 'ward' минимизирует дисперсию внутри объединяемых кластеров
linked = linkage(X_cluster, method="ward")

# 2. Визуализация дендрограммы
plt.figure(figsize=(16, 6))

# Уровень отсечения (Срез дерева)
CUT_DISTANCE = 30

# Построение дерева
dendrogram(
    linked,
    orientation="top",
    truncate_mode="level",
    p=5,
    distance_sort="descending",
    show_leaf_counts=True,
    leaf_rotation=90,
    leaf_font_size=10,
    color_threshold=CUT_DISTANCE,
)

# 3. Линия отсечения
plt.axhline(
    y=CUT_DISTANCE,
    color="crimson",
    linestyle="--",
    linewidth=2,
    label=f"Уровень отсечения (y={CUT_DISTANCE})",
)

plt.title("Иерархическая кластеризация: Дендрограмма (Метод Варда)", fontsize=14)
plt.xlabel("Индексы точек / (Количество точек в кластере)", fontsize=12)
plt.ylabel("Евклидово расстояние (Метод Варда)", fontsize=12)
plt.legend()
plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

#### **Анализ дендрограммы и выбор числа кластеров**

**1. Логика выбора уровня отсечения:**
* В иерархической кластеризации оптимальное количество сегментов определяется поиском наиболее длинных вертикальных ветвей на дендрограмме. Длина такой ветви (по оси Y) отражает степень "непохожести" между объединяемыми группами: чем она длиннее, тем более независимы эти кластеры друг от друга.
* На графике отчетливо виден значительный скачок расстояния (самый длинный участок без горизонтальных перемычек) в диапазоне от ~24 до 36 по оси Y. 
* Для фиксации сегментов горизонтальная пунктирная линия была проведена на уровне **`y = 30`**, что является математически надежным срезом на этом стабильном участке.

**2. Результаты разбиения:**
* Линия отсечения пересекает ровно **три** вертикальные ветви.
* Следовательно, оптимальным количеством признано **3 кластера**.

### **2.4.2. Применение иерархической кластеризации**

* __Цель:__ Извлечь финальные метки кластеров на основе выбранного среза дендрограммы и составить медицинские профили полученных групп.
* __Задачи:__ Провести «плоскую» кластеризацию дерева по заданному пороговому расстоянию для формирования финальных меток. Визуализировать итоговую трехкластерную структуру в 2D-проекции для оценки пространственной логики разделения. Доказать клиническую значимость выделенных групп путем анализа их средних медицинских профилей, включая «серую зону» умеренного риска.
* __Алгоритм использования:__
    1. __Формирование кластеров:__ Применение функции `fcluster` с использованием критерия `distance` и порогового значения $30$.
    2. __Контроль качества:__ Расчет итогового коэффициента силуэта для верификации плотности полученного разбиения.
    3. __Профилирование:__ Финальный расчет профилей средних значений для детального описания каждой группы пациентов.

In [ ]:
# --- 2.4.2.1. ПОЛУЧЕНИЕ МЕТОК КЛАСТЕРОВ ---

display(Markdown("#### **Получение меток иерархической кластеризации**"))

# Получение меток кластеров на основе выбранного уровня отсечения
CUT_DISTANCE = 30
hierarchical_labels = fcluster(linked, CUT_DISTANCE, criterion="distance")

# Расчет количества получившихся кластеров и коэффициента силуэта
n_clusters_hier = len(set(hierarchical_labels))
sil_score_hier = silhouette_score(X_cluster, hierarchical_labels)

display(
    Markdown(
        f"* Уровень отсечения дендрограммы (distance): **{CUT_DISTANCE}**\n"
        f"* Получено кластеров: **{n_clusters_hier}**\n"
        f"* Коэффициент силуэта: **{sil_score_hier:.4f}**"
    )
)

In [ ]:
# --- 2.4.2.2. ВИЗУАЛИЗАЦИЯ 2D-ПРОЕКЦИИ ---

display(Markdown("#### **Визуализация кластеров (Иерархия) в 2D-пространстве**"))

plot_df_hier = X_cluster.copy()
plot_df_hier["Cluster_Hierarchical"] = hierarchical_labels.astype(str)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    x="PC1",
    y="PC2",
    hue="Cluster_Hierarchical",
    palette="Set2",
    data=plot_df_hier,
    alpha=0.7,
    edgecolor="k",
    s=60,
)

plt.title("2D-проекция иерархической кластеризации (Метод Варда)", fontsize=14)
plt.xlabel("Главная компонента 1 (PC1)", fontsize=12)
plt.ylabel("Главная компонента 2 (PC2)", fontsize=12)
plt.legend(title="Кластер (Иерархия)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2.4.2.3. ИНТЕРПРЕТАЦИЯ КЛАСТЕРОВ ---

display(Markdown("#### **Анализ профилей сегментов (Иерархическая кластеризация)**"))

size_df_hier, means_hier = analyze_cluster_profiles(
    data_original, hierarchical_labels, "Cluster_Hierarchical"
)

#### **Выводы и интерпретация результатов (Иерархическая кластеризация)**

**1. Особенности иерархического разделения:**
* В отличие от K-Means (жесткое бинарное деление) и DBSCAN (выделение плотных ядер с отбрасыванием шума), метод Варда позволил изучить полную структуру вложенности данных от отдельных пациентов до единого кластера.
* Анализ дендрограммы математически обосновал наличие **трех** естественных сегментов. При этом алгоритм успешно классифицировал 100% пациентов, не потеряв ни одной записи в виде "шумовых" выбросов.

**2. Медицинский смысл сегментов (Выявление "серой зоны"):**
* Анализ средних показателей исходных признаков подтвердил формирование трех четких клинических профилей: пациенты с явной патологией (высокий риск) и условно здоровые люди (низкий риск) образовали два крайних полюса.
* Алгоритм успешно обособил **промежуточный кластер (группу умеренного риска)**. Это пациенты, чьи показатели (давление, макс. пульс, депрессия ST) находятся точно на границе между нормой и болезнью. То, что DBSCAN классифицировал как бесструктурный "шум", иерархический метод оформил в полноценную и крайне важную медицинскую группу.

**3. Сравнительный анализ:**
* Среди всех протестированных методов иерархическая кластеризация продемонстрировала **наиболее глубокий и клинически обоснованный результат**.
* Разделение на 3 сегмента дает максимально точную картину состояния популяции. В отличие от K-Means, который принудительно сглаживает пограничные случаи, и DBSCAN, который отсеивает их как шум, данный метод сохраняет всю полноту данных.
* Полученная топология позволяет сформировать строго дифференцированный медицинский подход: констатация нормы для первой группы, внимательное профилактическое наблюдение для "серой зоны" и назначение интенсивной терапии для группы подтвержденного риска.

In [ ]:
# --- ФОРМИРОВАНИЕ И СОХРАНЕНИЕ ДАННЫХ ---

data_clustered = data_original.copy()

data_clustered["Cluster_KMeans"] = cluster_labels
data_clustered["Cluster_DBSCAN"] = dbscan_labels
data_clustered["Cluster_Hierarchical"] = hierarchical_labels

save_dataset(data_clustered, "../data/processed", "heart_disease_uci_clustered.csv")

## **2.5. Сравнение моделей и общие выводы**

### **2.5.1. Сводная оценка алгоритмов кластеризации**

1. __Результаты исследования:__
В ходе анализа кардиологических данных были применены три различных алгоритма кластеризации без учителя. Исходная гипотеза предполагала строго бинарное разделение пациентов (здоровые и больные). Однако многомерный анализ выявил более сложную внутреннюю топологию данных.
2. __Оценка алгоритмов:__
   * **K-Means** успешно справился с базовым разделением, но принудительно сгладил пограничные состояния пациентов, раскидав их по двум крайним полюсам.
   * **DBSCAN** наглядно доказал наличие обширной переходной зоны. Из-за отсутствия четкой плотности в этой зоне алгоритм классифицировал значительную часть пациентов как "шум".
   * **Иерархическая кластеризация (Метод Варда)** показала наилучший аналитический результат. Анализ дендрограммы позволил математически обосновать наличие **трех кластеров**, сохранив при этом 100% данных и сформировав полноценный промежуточный профиль.
3. __Сравнительный анализ алгоритмов кластеризации:__
      
      | Метод кластеризации | Количество кластеров | Коэффициент силуэта | Количество шумовых точек |
      | :--- | :---: | :---: | :---: |
      | **K-Means** | 2 | 0.1767 | — |
      | **DBSCAN** | 2 | — | 370 (40.3%) |
      | **Иерархическая (Ward)** | 3 | 0.1595 | 0 |

---

### **2.5.2. Итоговые сегменты пациентов**

На основе комплексного применения алгоритмов (в частности, иерархического метода) база пациентов была сегментирована на 3 итоговых кластера, имеющих четкий медицинский профиль:

* __Сегмент А: "Зеленая зона" (Низкий риск)__
  * __Характеристики:__ Самая молодая группа (средний возраст ~47 лет). Нормальное давление, отличная переносимость физических нагрузок, отсутствие стенокардии. 

* __Сегмент B: "Желтая зона" (Умеренный риск / Пограничное состояние)__
  * __Характеристики:__ Пациенты, чьи показатели начинают отклоняться от нормы (появляются первые скачки давления или снижается максимальный пульс), но критических патологий еще нет. 

* __Сегмент C: "Красная зона" (Высокий риск / Патология)__
  * __Характеристики:__ Возрастные клиенты (~59 лет) с явной симптоматикой: повышенное давление в покое, стенокардия при малейшей нагрузке, плохие показатели ЭКГ (депрессия ST).